# Claims Anomaly Detection

**Objective:** Detect unusual claims patterns using statistical and ML-based
anomaly detection methods. In healthcare data ops, anomaly detection
is critical for:

- **Fraud** — providers billing for services not rendered
- **Abuse** — excessive utilization (unnecessary procedures, overbilling)
- **Errors** — billing system glitches, duplicate claims
- **Clinical flags** — unusual diagnosis combinations

**Methods used:**
1. **Z-score** — flag claims >3σ from mean (cost-based)
2. **IQR method** — non-parametric outlier detection
3. **Isolation Forest** — ML-based multivariate anomaly detection
4. **DBSCAN clustering** — density-based outlier detection

**Why this matters:** In value-based care, anomalous claims don't just
waste money — they can indicate patient safety risks, billing fraud,
or systematic errors that affect care quality.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('data/claims_dirty.csv')
print(f"Total claims: {len(df)}")
print(f"Unique members: {df['mem_id'].nunique()}")
print(f"Date range: {df['claim_date'].min()} → {df['claim_date'].max()}")
df.head()

## Method 1 — Z-Score (Univariate Cost Outlier)

Flag claims where the paid amount is more than 3 standard deviations
above the mean. Simple, fast, interpretable.

In [ ]:
# Z-score method
df['z_score'] = np.abs(stats.zscore(df['paid_amount']))
df['z_flag'] = (df['z_score'] > 3).astype(int)

z_outliers = df[df['z_flag'] == 1]
print(f"⚠️  Z-score outliers (>3σ): {len(z_outliers)} claims")
print(f"   Total flagged amount: ${z_outliers['paid_amount'].sum():,.2f}")
print(f"\nExamples:")
print(z_outliers[['claim_id','mem_id','paid_amount','specialty','claim_date']].head(10).to_string())

## Method 2 — IQR (Non-Parametric)

Q1 - 1.5×IQR and Q3 + 1.5×IQR as bounds. Works well for
skewed (non-normal) cost distributions.

In [ ]:
# IQR method
Q1 = df['paid_amount'].quantile(0.25)
Q3 = df['paid_amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['iqr_flag'] = ((df['paid_amount'] < lower) | (df['paid_amount'] > upper)).astype(int)
iqr_outliers = df[df['iqr_flag'] == 1]
print(f"IQR bounds: ${lower:.2f} – ${upper:.2f}")
print(f"⚠️  IQR outliers: {len(iqr_outliers)} claims")
print(f"   Percentage of total: {100*len(iqr_outliers)/len(df):.1f}%")

## Method 3 — Isolation Forest (Multivariate ML)

Uses multiple features simultaneously: paid amount, copay ratio,
member claim frequency. Catches patterns invisible to univariate methods.

In [ ]:
# Build member-level features first
member_claims = df.groupby('mem_id').agg(
    total_paid=('paid_amount', 'sum'),
    claim_count=('claim_id', 'count'),
    avg_paid=('paid_amount', 'mean'),
    std_paid=('paid_amount', 'std'),
    avg_copay_ratio=('member_copay', lambda x: (x / df.loc[x.index, 'allowed_amount']).mean())
).reset_index()
member_claims['std_paid'] = member_claims['std_paid'].fillna(0)

# Select features for Isolation Forest
features = ['total_paid', 'claim_count', 'avg_paid', 'std_paid']
X = member_claims[features].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # Expect ~5% anomalies
    random_state=42,
    n_estimators=100
)
member_claims['anomaly_score'] = iso_forest.fit_predict(X_scaled)
member_claims['anomaly_label'] = (member_claims['anomaly_score'] == -1).astype(int)

iso_anomalies = member_claims[member_claims['anomaly_label'] == 1]
print(f"🌲 Isolation Forest anomalies: {len(iso_anomalies)} members")
print(f"\nTop anomalous members (highest anomaly score):")
top_anomalies = iso_anomalies.sort_values('total_paid', ascending=False)
print(top_anomalies[['mem_id','total_paid','claim_count','avg_paid','std_paid']].head(15).to_string())

## Method 4 — Utilization Pattern Anomaly

Flag members whose claim frequency is statistically unusual.
Most members have 10-20 claims/year. Flag those with 30+.

In [ ]:
# Utilization anomaly
mean_claims = member_claims['claim_count'].mean()
std_claims = member_claims['claim_count'].std()
threshold = mean_claims + 2.5 * std_claims

member_claims['utilization_flag'] = (member_claims['claim_count'] > threshold).astype(int)

high_util = member_claims[member_claims['utilization_flag'] == 1].sort_values('claim_count', ascending=False)
print(f"📊 Mean claims/member: {mean_claims:.1f}")
print(f"   Std: {std_claims:.1f}")
print(f"   Flag threshold: {threshold:.1f}")
print(f"\n⚠️  High utilization members: {len(high_util)}")
print(high_util[['mem_id','claim_count','total_paid','avg_paid']].head(15).to_string())

## Consolidated Anomaly Report

Merge all detection methods and rank members by risk.

In [ ]:
# Merge all flags
df_merged = df.merge(member_claims[['mem_id','claim_count','utilization_flag','anomaly_label']], on='mem_id')

# Member-level risk score
member_risk = member_claims.copy()
member_risk['risk_score'] = 0

# +1 for high utilization
member_risk['risk_score'] += member_risk['utilization_flag']

# +1 for isolation forest anomaly
member_risk['risk_score'] += member_risk['anomaly_label']

# +1 for having any z-score flagged claim
z_flagged_members = df[df['z_flag'] == 1]['mem_id'].unique()
member_risk['risk_score'] += member_risk['mem_id'].isin(z_flagged_members).astype(int)

# Risk tiers
member_risk['risk_tier'] = pd.cut(
    member_risk['risk_score'],
    bins=[-1, 0, 1, 2, 99],
    labels=['✅ Normal', '🟡 Monitor', '🟠 Elevated', '🔴 Critical']
)

print("RISK TIER DISTRIBUTION:\n")
print(member_risk['risk_tier'].value_counts())

critical = member_risk[member_risk['risk_tier'] == '🔴 Critical']
print(f"\n🔴 CRITICAL RISK MEMBERS: {len(critical)}")
if len(critical) > 0:
    print(critical[['mem_id','claim_count','total_paid','risk_score']].sort_values('total_paid', ascending=False).to_string())

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cost distribution with outliers
ax1 = axes[0]
normal = df[df['z_flag'] == 0]['paid_amount']
outliers = df[df['z_flag'] == 1]['paid_amount']
ax1.hist(normal, bins=50, alpha=0.7, label=f'Normal ({len(normal)})')
ax1.hist(outliers, bins=20, alpha=0.9, color='red', label=f'Outliers ({len(outliers)})')
ax1.set_xlabel('Paid Amount ($)')
ax1.set_ylabel('Frequency')
ax1.set_title('Claims Cost Distribution — Z-Score Outliers in Red')
ax1.legend()

# Plot 2: Utilization scatter
ax2 = axes[1]
colors = member_risk['risk_tier'].map({
    '✅ Normal': 'green',
    '🟡 Monitor': 'yellow',
    '🟠 Elevated': 'orange',
    '🔴 Critical': 'red'
})
ax2.scatter(member_risk['claim_count'], member_risk['total_paid'],
           c=colors, alpha=0.6, s=30)
ax2.set_xlabel('Claim Count')
ax2.set_ylabel('Total Paid ($)')
ax2.set_title('Member Risk Map — Utilization vs Cost')

plt.tight_layout()
plt.savefig('../data/anomaly_detection_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Saved: data/anomaly_detection_results.png")

## Export Results

In [ ]:
# Export flagged claims
flagged_claims = df_merged[
    (df_merged['z_flag'] == 1) | 
    (df_merged['iqr_flag'] == 1) |
    (df_merged['anomaly_label'] == 1) |
    (df_merged['utilization_flag'] == 1)
].sort_values('paid_amount', ascending=False)

flagged_claims.to_csv('../data/flagged_claims.csv', index=False)

# Export member risk report
member_risk.to_csv('../data/member_risk_report.csv', index=False)

print(f"🚨 Flagged claims: {len(flagged_claims)} → data/flagged_claims.csv")
print(f"📋 Member risk report: {len(member_risk)} members → data/member_risk_report.csv")

# Summary
print("\n" + "="*60)
print("ANOMALY DETECTION SUMMARY")
print("="*60)
print(f"  Total claims analyzed: {len(df)}")
print(f"  Unique members: {df['mem_id'].nunique()}")
print(f"  Z-score outliers (>3σ): {df['z_flag'].sum()}")
print(f"  IQR outliers: {df['iqr_flag'].sum()}")
print(f"  Isolation Forest anomalies: {member_risk['anomaly_label'].sum()} members")
print(f"  High utilization: {member_risk['utilization_flag'].sum()} members")
print(f"  🔴 Critical risk: {(member_risk['risk_tier'] == '🔴 Critical').sum()} members")